# FINN3021 — Ex-ante Equity Screener

**Objective:** build a 20-name institutional-quality equity portfolio from MSCI DM ex-US + EM using absolute-bar gates, a quality/persistence composite, structural under-pricing tags, and a 2-year survivability stress test.

**Philosophy:** *Quality at a reasonable price* — cheapness is a gate (pass/fail), not a ranking input. The composite ranks purely on quality and persistence of returns.

**Data source:** WRDS Compustat Global (`comp.g_company`, `comp.g_funda`, `comp.g_secd`).

---

| Stage | What it does |
|-------|-------------|
| 1. Universe & data pull | 46 ex-US countries, exclude Financials + Real Estate |
| 2. Derived metrics | FCF, ROIC, EBITDA, net debt — per firm-year panel |
| 3. Hard gates | Size, solvency, quality, and valuation floors |
| 4. Composite scoring | `quality^0.6 x persistence^0.4` (geometric) |
| 5. Advisory flags | 9-char trap-flag string (SRACNDLXV) |
| 6. Structural tags | 7 under-pricing indicators |
| 7. Selection | Top 20 zero-flag names with country + sector caps |
| 8. Stress test | Base / stress / severe scenarios; survivability ratings |
| 9. Visualisations | Diagnostics, bar/radar charts, price histories |

## 0 - Setup

In [ ]:
%pip install --quiet wrds pandas numpy matplotlib seaborn python-dotenv pyarrow yfinance requests tabulate

In [ ]:
import os, pathlib, json
from datetime import date
from dataclasses import dataclass, field
from typing import Any, Iterable
from pandas.tseries.offsets import DateOffset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from tabulate import tabulate

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*a, **k): pass

ROOT = pathlib.Path().resolve().parent
load_dotenv(ROOT / ".env")
CACHE = ROOT / "data" / "cache"; CACHE.mkdir(parents=True, exist_ok=True)
OUT   = ROOT / "data" / "outputs"; OUT.mkdir(parents=True, exist_ok=True)

print(f"run at: {pd.Timestamp.now(tz='UTC').isoformat(timespec='seconds')}")

## 1 - Screen Parameters

All tuneable constants live here. Change once, re-run notebook.

Key design choices:
- **Tiered market-cap floors**: DM $2bn / EM $1bn (institutional liquidity)
- **Absolute quality gates**: ROIC, FCF margin, Piotroski — pass/fail, not relative ranking
- **4-way valuation gate**: EY > 5% OR FCF/EV > 5% OR maint-FCF/EV > 5% OR (ROIC >= 15% AND EY > 3%)
- **Composite = quality^0.6 x persistence^0.4** (cheapness excluded from ranking)
- **Concentration caps**: max 25% per country AND per GICS sector

In [ ]:
SCREEN_ASOF = date(2026, 3, 9)

# ── Universe ──────────────────────────────────────────────────────────
DM_EX_US = {"GB","JP","CA","AU","DE","FR","CH","NL","SE","ES","IT","DK",
            "HK","SG","BE","NO","FI","IL","IE","AT","PT","NZ"}
EM       = {"CN","IN","KR","TW","BR","ZA","MX","SA","AE","QA","KW","TH",
            "MY","ID","PH","PL","HU","CZ","GR","TR","CL","PE","CO","EG"}
UNIVERSE = DM_EX_US | EM
EXCLUDED_GSECTOR = {"40", "60"}   # Financials, Real Estate

# ── Size / liquidity ──────────────────────────────────────────────────
MCAP_MIN_USD_DM    = 2_000_000_000
MCAP_MIN_USD_EM    = 1_000_000_000
ADV_MIN_USD        = 100_000
ADV_FLAG_THRESHOLD = 1_000_000

# ── Absolute quality gates ────────────────────────────────────────────
ROIC_5Y_MIN        = 0.10
ROIC_10Y_MIN       = 0.05
FCF_MARGIN_5Y_MIN  = 0.05
PIOTROSKI_MIN      = 5

# ── Valuation gate ────────────────────────────────────────────────────
EY_MIN             = 0.05
FCF_EV_MIN         = 0.05
ROIC_ESCAPE_MIN    = 0.15    # proven compounder bypass
EY_ESCAPE_MIN      = 0.03    # ... at PE up to ~33x
FCF_EV_FLAG_CEIL   = 0.15
SHARES_DILUTION_FLAG = 0.05
MIN_EV_MCAP_RATIO  = 0.30
FCF_EV_CAP         = 0.60

# ── Advisory flag thresholds ──────────────────────────────────────────
SALES_CAGR_MIN     = -0.02
ROIC_RATIO_MIN     = 0.70
ACCRUALS_MAX       = 0.05
CYCLICAL_GSECTORS  = {"10", "15"}
FCF_CAGR_MAX_NORMAL = 1.00
ROIC_SURGE_MULT    = 2.00

# ── Selection & concentration ─────────────────────────────────────────
N_FINAL            = 20
COUNTRY_MAX_PCT    = 0.25
SECTOR_MAX_PCT     = 0.25
N_YEARS_FUNDA      = 5
FCF_HISTORY_YEARS  = 5
LEVERAGE_MAX       = 4.0

# ── Structural under-pricing ──────────────────────────────────────────
STRUCT_PURE_WEIGHT   = 0.20
STRUCT_MIXED_WEIGHT  = 0.10
STRUCT_COVERAGE_MCAP_FLOOR = 1_500_000_000
STRUCT_ESG_DEEP      = {30203010, 25201010, 25301010, 10102050}
STRUCT_ESG_PARTIAL   = {20101010, 30201020, 30201030, 10102010, 10102020}
STRUCT_DM_COUNTRIES  = {"JP","GB","DE","FR","CH","AU","CA","SE","DK","NO","FI","NL","BE",
                        "IT","ES","IE","HK","SG","IL","NZ","AT","PT"}
STRUCT_ORPHAN_COUNTRY = {"SA","VN","PK","EG","GR","HU","CZ","PE","CL","CO","PH",
                         "AE","QA","KW","TR","MA"}
STRUCT_RETAIL_DOMINATED = {"KR","TW","IN","TR","TH","VN"}
STRUCT_BMK_DM_LO, STRUCT_BMK_DM_HI = 500_000_000, 3_000_000_000
STRUCT_BMK_EM_LO, STRUCT_BMK_EM_HI = 300_000_000, 2_000_000_000

# ── QA limits ─────────────────────────────────────────────────────────
QA_MAX_ROIC = 3.0; QA_MIN_ROIC = -3.0; QA_MAX_LEVERAGE = 50.0
CN_WHITELIST_PATTERNS = {"BYD"}

print(f"Screen as-of: {SCREEN_ASOF}")
print(f"Universe: {len(UNIVERSE)} countries ({len(DM_EX_US)} DM + {len(EM)} EM)")

## 2 - WRDS Connection & Data Pull

Connects to WRDS Postgres (preferred) with parquet caching. Pulls:
- `comp.g_company` — firm identifiers, country, GICS classification
- `comp.g_funda` — annual fundamentals (10 years)
- `comp.g_secd` — daily security prices (for EV, mcap, liquidity)

In [ ]:
WRDS_USERNAME  = os.getenv("WRDS_USERNAME")
WRDS_API_TOKEN = os.getenv("WRDS_API_TOKEN")

db = None
try:
    import wrds
    db = wrds.Connection(wrds_username=WRDS_USERNAME)
    print(f"[wrds] Postgres connected as {WRDS_USERNAME}")
except Exception as e:
    print(f"[wrds] Postgres unavailable ({e}); cannot proceed without connection")

def sql_cached(query: str, cache_name: str, force: bool = False) -> pd.DataFrame:
    path = CACHE / f"{cache_name}.parquet"
    if path.exists() and not force:
        print(f"[cache hit] {cache_name}")
        return pd.read_parquet(path)
    if db is None:
        raise RuntimeError("No Postgres connection.")
    print(f"[sql pull] {cache_name} ...")
    df = db.raw_sql(query)
    df.to_parquet(path, index=False)
    print(f"           {len(df):,} rows -> cache/{cache_name}.parquet")
    return df

In [ ]:
# ── ISO-2 to ISO-3 mapping (Compustat Global uses ISO-3 for fic) ──────
ISO2_TO_ISO3 = {
    "AE":"ARE","AT":"AUT","AU":"AUS","BE":"BEL","BR":"BRA","CA":"CAN","CH":"CHE",
    "CL":"CHL","CN":"CHN","CO":"COL","CZ":"CZE","DE":"DEU","DK":"DNK","EG":"EGY",
    "ES":"ESP","FI":"FIN","FR":"FRA","GB":"GBR","GR":"GRC","HK":"HKG","HU":"HUN",
    "ID":"IDN","IE":"IRL","IL":"ISR","IN":"IND","IT":"ITA","JP":"JPN","KR":"KOR",
    "KW":"KWT","MX":"MEX","MY":"MYS","NL":"NLD","NO":"NOR","NZ":"NZL","PE":"PER",
    "PH":"PHL","PL":"POL","PT":"PRT","QA":"QAT","SA":"SAU","SE":"SWE","SG":"SGP",
    "TH":"THA","TR":"TUR","TW":"TWN","ZA":"ZAF",
}
ISO3_TO_ISO2 = {v: k for k, v in ISO2_TO_ISO3.items()}
UNIVERSE_ISO3 = {ISO2_TO_ISO3[c] for c in UNIVERSE}
iso_sql = ",".join(f"'{c}'" for c in sorted(UNIVERSE_ISO3))

# ── Company table ─────────────────────────────────────────────────────
q_company = f"""
    SELECT gvkey, conm AS company_name, fic, gsector, ggroup, gind, gsubind, loc, ipodate
    FROM comp.g_company
    WHERE fic IN ({iso_sql})
"""
company = sql_cached(q_company, "g_company_universe")
company = company[~company["gsector"].astype(str).isin(EXCLUDED_GSECTOR)].copy()
print(f"Company universe: {len(company):,} firms across {company['fic'].nunique()} countries")

In [ ]:
# ── Annual fundamentals ────────────────────────────────────────────────
excl_sector_sql = ",".join(f"'{s}'" for s in EXCLUDED_GSECTOR)
q_funda = f"""
    SELECT f.gvkey, f.datadate, f.fyear,
           f.oancf, f.capx, f.dp, f.oibdp, f.ebit,
           f.dlc, f.dltt, f.che,
           f.nicon, f.at, f.act, f.lct, f.cshoi, f.sale, f.cogs,
           f.ceq, f.txt, f.pi, f.curcd,
           f.dvt, f.xint, f.re, f.lt
    FROM comp.g_funda f
    JOIN comp.g_company c USING (gvkey)
    WHERE c.fic IN ({iso_sql})
      AND NOT c.gsector::text IN ({excl_sector_sql})
      AND f.indfmt = 'INDL' AND f.datafmt = 'HIST_STD'
      AND f.consol = 'C'    AND f.popsrc  = 'I'
      AND f.datadate BETWEEN '2016-01-01' AND '{SCREEN_ASOF}'
"""
funda_raw = sql_cached(q_funda, "g_funda_raw_v4")
print(f"Fundamentals: {len(funda_raw):,} rows | {funda_raw['gvkey'].nunique():,} firms")

# Clean & apply point-in-time filter
funda = funda_raw.copy().rename(columns={"nicon": "ni", "cshoi": "csho"})
funda["datadate"]       = pd.to_datetime(funda["datadate"])
funda["effective_date"] = funda["datadate"]
funda = funda[funda["effective_date"] <= pd.Timestamp(SCREEN_ASOF)].copy()
funda = funda.merge(company[["gvkey", "fic", "gsector"]], on="gvkey", how="inner")
print(f"After point-in-time filter: {len(funda):,} rows")

## 3 - Derived Per-Firm-Year Metrics

Core calculations applied to every firm-year:
- **FCF** = Operating cash flow - CapEx
- **ROIC** = EBIT x (1 - 25%) / max(Total Assets - Current Liabilities, 20% of TA) — capped at 60%
- **Net debt** = Short-term debt + Long-term debt - Cash

In [ ]:
p = funda.copy()
p["fcf"]      = p["oancf"] - p["capx"]
p["ebitda"]   = p["oibdp"]
p["net_debt"] = p["dlc"].fillna(0) + p["dltt"].fillna(0) - p["che"].fillna(0)

# ROIC: flat 25% tax, invested capital = AT - current liabilities, 20% AT floor, 60% cap
tax_rate = 0.25
p["tax_rate"] = tax_rate
nopat = p["ebit"] * (1 - tax_rate)
invested = pd.to_numeric(p["at"], errors="coerce") - pd.to_numeric(p["lct"], errors="coerce").fillna(0)
invested = invested.clip(lower=0.20 * pd.to_numeric(p["at"], errors="coerce"))
p["roic"] = (nopat / invested.where(invested > 0)).clip(upper=0.60)

panel = p
print(f"Panel: {len(panel):,} firm-years | {panel['gvkey'].nunique():,} firms")
print(f"ROIC: median={panel['roic'].median():.1%}, p90={panel['roic'].quantile(0.9):.1%}")

## 4 - Tier 1: History, FCF Consistency & Leverage

Require at least 5 years of history, all-positive FCF, and net debt/EBITDA < 4x.

In [ ]:
fy_counts = panel.groupby("gvkey")["fyear"].nunique()
firms_with_history = fy_counts[fy_counts >= FCF_HISTORY_YEARS].index
pH = panel[panel["gvkey"].isin(firms_with_history)].copy()
pH = pH.sort_values(["gvkey", "fyear"]).groupby("gvkey").tail(N_YEARS_FUNDA)

fcf_pos_all = pH.groupby("gvkey")["fcf"].apply(lambda s: (s > 0).all())
firms_fcf_ok = fcf_pos_all[fcf_pos_all].index

latest = pH.sort_values("fyear").groupby("gvkey").tail(1).set_index("gvkey")
latest["net_debt_ebitda"] = latest["net_debt"] / latest["ebitda"].where(latest["ebitda"] > 0)
firms_lev_ok = latest[latest["net_debt_ebitda"] < LEVERAGE_MAX].index

survivors = set(firms_fcf_ok) & set(firms_lev_ok)
print(f"  >= {FCF_HISTORY_YEARS} FYs of history: {len(firms_with_history):,}")
print(f"  All {N_YEARS_FUNDA} FYs FCF > 0:      {len(firms_fcf_ok):,}")
print(f"  Net debt/EBITDA < {LEVERAGE_MAX}x:  {len(firms_lev_ok):,}")
print(f"  -> TIER 1 SURVIVORS:        {len(survivors):,}")

panel5 = pH[pH["gvkey"].isin(survivors)].copy()

### Market Data: Prices, Market Cap, EV, Liquidity

Pull daily security prices, compute point-in-time market cap (at screen date), enterprise value per fiscal year, and 60-day average dollar volume.

In [ ]:
# ── Security prices ────────────────────────────────────────────────────
gv_list = ",".join(f"'{g}'" for g in panel5["gvkey"].unique())
q_sec = f"""
    SELECT gvkey, datadate, prccd, cshoc, divd, cshtrd
    FROM comp.g_secd
    WHERE gvkey IN ({gv_list})
      AND datadate BETWEEN '2018-01-01' AND '{SCREEN_ASOF}'
"""
sec = sql_cached(q_sec, "g_secd_survivors_v4")
sec["datadate"] = pd.to_datetime(sec["datadate"])

# De-duplicate (some firms have multiple issues)
sec = (sec.groupby(["gvkey", "datadate"], as_index=False)
          .agg(prccd=("prccd","median"), cshoc=("cshoc","median"),
               divd=("divd","median"), cshtrd=("cshtrd","median")))

# ── Fiscal year-end EV ─────────────────────────────────────────────────
sec_sorted = sec.sort_values("datadate")
fye_keys = (panel5[["gvkey", "fyear", "datadate", "csho", "net_debt"]]
            .drop_duplicates().sort_values("datadate"))
fye_prices = pd.merge_asof(
    fye_keys, sec_sorted[["gvkey", "datadate", "prccd"]],
    on="datadate", by="gvkey", direction="backward"
).rename(columns={"prccd": "price_fye"})

fye_prices["market_cap"] = fye_prices["price_fye"] * fye_prices["csho"]
fye_prices["ev_raw"]     = fye_prices["market_cap"] + fye_prices["net_debt"]
pass_gate = ((fye_prices["ev_raw"] > 0) & (fye_prices["market_cap"] > 0) &
             (fye_prices["ev_raw"] >= MIN_EV_MCAP_RATIO * fye_prices["market_cap"]))
fye_prices["ev"] = fye_prices["ev_raw"].where(pass_gate)
ev_by_year = fye_prices[["gvkey", "fyear", "market_cap", "ev"]].copy()

# ── Point-in-time market cap at SCREEN_ASOF ────────────────────────────
sec_asof = (sec.sort_values("datadate").groupby("gvkey").tail(1)
              [["gvkey", "datadate", "prccd", "cshoc"]]
              .rename(columns={"datadate": "asof_price_date",
                               "prccd": "price_asof", "cshoc": "cshoc_asof"}))
sec_asof["market_cap_local"] = sec_asof["price_asof"] * sec_asof["cshoc_asof"]
latest_mcap = sec_asof.merge(panel5.drop_duplicates("gvkey")[["gvkey", "curcd"]], on="gvkey", how="left")

# ── Static FX table (USD per 1 unit of local currency, ~Q1 2026) ──────
usd_per_local = pd.Series({
    "USD":1.0, "EUR":1.08, "GBP":1.27, "JPY":0.0067, "CHF":1.13,
    "CAD":0.74, "AUD":0.66, "NZD":0.61, "SEK":0.096, "NOK":0.095,
    "DKK":0.145, "CZK":0.0435, "PLN":0.252, "HUF":0.0027, "TRY":0.028,
    "ILS":0.275, "SAR":0.2667, "AED":0.2723, "QAR":0.2747, "KWD":3.25,
    "ZAR":0.0545, "EGP":0.0205, "INR":0.012, "CNY":0.1385, "HKD":0.1282,
    "TWD":0.0315, "KRW":0.00075, "SGD":0.75, "MYR":0.226, "THB":0.0285,
    "IDR":0.000062, "PHP":0.0178, "BRL":0.172, "MXN":0.049, "CLP":0.00103,
    "COP":0.000245, "PEN":0.268, "RUB":0.0108,
})
latest_mcap["fx_to_usd"]      = latest_mcap["curcd"].map(usd_per_local)
latest_mcap["market_cap_usd"] = latest_mcap["market_cap_local"] * latest_mcap["fx_to_usd"]

# ── TTM dividend per share ─────────────────────────────────────────────
_sec_div = sec[["gvkey","datadate","divd"]].dropna(subset=["divd"])
_sec_div = _sec_div[_sec_div["divd"] > 0].merge(latest_mcap[["gvkey","asof_price_date"]], on="gvkey")
_window_lo = _sec_div["asof_price_date"] - pd.Timedelta(days=365)
_in_window = (_sec_div["datadate"] > _window_lo) & (_sec_div["datadate"] <= _sec_div["asof_price_date"])
_ttm_dps = _sec_div[_in_window].groupby("gvkey")["divd"].sum().rename("ttm_dps").reset_index()
latest_mcap = latest_mcap.merge(_ttm_dps, on="gvkey", how="left")
latest_mcap["ttm_dps"] = latest_mcap["ttm_dps"].fillna(0.0)

# ── 60-day average daily dollar volume (ADV) ──────────────────────────
_sv = sec[["gvkey","datadate","prccd","cshtrd"]].merge(
    latest_mcap[["gvkey","asof_price_date"]], on="gvkey", how="inner")
_sv = _sv[_sv["datadate"] <= _sv["asof_price_date"]]
_sv = _sv.sort_values(["gvkey","datadate"]).groupby("gvkey").tail(60)
_sv["dollar_vol_local"] = pd.to_numeric(_sv["prccd"], errors="coerce") * pd.to_numeric(_sv["cshtrd"], errors="coerce")
_adv_local = _sv.groupby("gvkey")["dollar_vol_local"].mean().rename("adv_local").reset_index()
latest_mcap = latest_mcap.merge(_adv_local, on="gvkey", how="left")
latest_mcap["adv_usd"] = latest_mcap["adv_local"] * latest_mcap["fx_to_usd"]

print(f"Market cap (USD): median ${latest_mcap['market_cap_usd'].median()/1e6:.0f}m")
print(f"ADV (USD): median ${latest_mcap['adv_usd'].median()/1e3:.0f}k")

## 5 - Firm-Level Scoring Metrics

Piotroski F-score, ROIC/FCF growth, FCF/EV, maintenance-FCF yield, persistence metrics (10y), advisory flags.

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────
def _num(x):
    try:
        if pd.isna(x): return np.nan
        return float(x)
    except (TypeError, ValueError):
        return np.nan

def _gt(a, b):
    a, b = _num(a), _num(b)
    return int((not np.isnan(a)) and (not np.isnan(b)) and a > b)

def _lt(a, b):
    a, b = _num(a), _num(b)
    return int((not np.isnan(a)) and (not np.isnan(b)) and a < b)

def _le(a, b):
    a, b = _num(a), _num(b)
    return int((not np.isnan(a)) and (not np.isnan(b)) and a <= b)

def _safe_div(n, d):
    n, d = _num(n), _num(d)
    if np.isnan(n) or np.isnan(d) or d == 0: return np.nan
    return n / d

# ── Piotroski F-score (9 factors) ─────────────────────────────────────
def piotroski_fscore(panel5: pd.DataFrame) -> pd.Series:
    pp = panel5.sort_values(["gvkey", "fyear"])
    scores = {}
    for gv, g in pp.groupby("gvkey"):
        if len(g) < 2: continue
        t, tm1 = g.iloc[-1], g.iloc[-2]
        s = 0
        s += _gt(t["ni"], 0)
        s += _gt(t["oancf"], 0)
        s += _gt(_safe_div(t["ni"], t["at"]), _safe_div(tm1["ni"], tm1["at"]))
        s += _gt(t["oancf"], t["ni"])
        s += _lt(_safe_div(t["dltt"], t["at"]), _safe_div(tm1["dltt"], tm1["at"]))
        s += _gt(_safe_div(t["act"], t["lct"]), _safe_div(tm1["act"], tm1["lct"]))
        s += _le(t["csho"], tm1["csho"])
        gm_t   = _safe_div(_num(t["sale"]) - _num(t["cogs"]), t["sale"])
        gm_tm1 = _safe_div(_num(tm1["sale"]) - _num(tm1["cogs"]), tm1["sale"])
        s += _gt(gm_t, gm_tm1)
        s += _gt(_safe_div(t["sale"], t["at"]), _safe_div(tm1["sale"], tm1["at"]))
        scores[gv] = s
    return pd.Series(scores, name="piotroski")

# ── Growth & valuation metrics ────────────────────────────────────────
roic_5y = panel5.groupby("gvkey")["roic"].mean().rename("roic_5y_avg")

def _cagr(s: pd.Series) -> float:
    s = pd.to_numeric(s, errors="coerce").dropna().sort_index()
    if len(s) < N_YEARS_FUNDA: return np.nan
    first, last = float(s.iloc[0]), float(s.iloc[-1])
    if first <= 0 or last <= 0: return np.nan
    return (last / first) ** (1/(N_YEARS_FUNDA - 1)) - 1

fcfg_5y   = panel5.set_index("fyear").groupby("gvkey")["fcf"].apply(_cagr).rename("fcf_growth_5y")
salesg_5y = panel5.set_index("fyear").groupby("gvkey")["sale"].apply(_cagr).rename("sales_growth_5y")

# ── FCF/EV (winsorized) ───────────────────────────────────────────────
ev_merge = panel5.merge(ev_by_year, on=["gvkey", "fyear"], how="left")
ev_merge["fcf_ev"] = ev_merge["fcf"] / ev_merge["ev"].where(ev_merge["ev"] > 0)
ev_merge["fcf_ev_wins"] = ev_merge["fcf_ev"].clip(upper=FCF_EV_CAP)
fev_5y = ev_merge.groupby("gvkey")["fcf_ev_wins"].mean().rename("fcf_ev_5y_avg")

# ── Maintenance-FCF yield (owner earnings) ────────────────────────────
# maint_fcf = oancf - max(D&A, 5y-min capex); only penalises maintenance capex
capx_5y_min = panel5.groupby("gvkey")["capx"].apply(
    lambda s: pd.to_numeric(s, errors="coerce").min()).rename("capx_5y_min")
ev_merge2 = ev_merge.merge(capx_5y_min, on="gvkey", how="left")
_dp_s   = pd.to_numeric(ev_merge2["dp"], errors="coerce")
_cmin_s = pd.to_numeric(ev_merge2["capx_5y_min"], errors="coerce").fillna(0)
_oan_s  = pd.to_numeric(ev_merge2["oancf"], errors="coerce")
_maint_capx = np.where(_dp_s.notna(), np.maximum(_dp_s.fillna(0).values, _cmin_s.values), np.nan)
ev_merge2["maint_fcf"]    = _oan_s - _maint_capx
ev_merge2["maint_fcf_ev"] = ev_merge2["maint_fcf"] / ev_merge2["ev"].where(ev_merge2["ev"] > 0)
ev_merge2["maint_fcf_ev_wins"] = ev_merge2["maint_fcf_ev"].clip(upper=FCF_EV_CAP)
maint_fev_5y = ev_merge2.groupby("gvkey")["maint_fcf_ev_wins"].mean().rename("maint_fcf_ev_5y_avg")

# Growth-capex flag: FYs where capx > 1.5x D&A
def _growth_capx_years(g):
    g = g.sort_values("fyear").tail(5)
    dp_v = pd.to_numeric(g["dp"], errors="coerce")
    cx_v = pd.to_numeric(g["capx"], errors="coerce")
    ok = (dp_v > 0) & cx_v.notna()
    if ok.sum() == 0: return 0
    return int((cx_v[ok] > 1.5 * dp_v[ok]).sum())
growth_capx_years_5y = panel5.groupby("gvkey").apply(_growth_capx_years).rename("growth_capx_years_5y")

# ── ROIC deterioration & accruals ─────────────────────────────────────
latest_by_gv = panel5.sort_values("fyear").groupby("gvkey").tail(1).set_index("gvkey")
roic_latest  = latest_by_gv["roic"].rename("roic_latest")
roic_ratio   = (roic_latest / roic_5y).rename("roic_ratio_latest_5y")
last2 = panel5.sort_values("fyear").groupby("gvkey").tail(2)
avg_at = last2.groupby("gvkey")["at"].mean().rename("avg_at_2y")
accruals = ((latest_by_gv["ni"] - latest_by_gv["oancf"]) / avg_at).rename("accruals_latest")

pio = piotroski_fscore(panel5)
fic_by_gv     = panel5.drop_duplicates("gvkey").set_index("gvkey")["fic"]
gsector_by_gv = panel5.drop_duplicates("gvkey").set_index("gvkey")["gsector"].rename("gsector")

# ── Assemble metrics DataFrame ────────────────────────────────────────
metrics = (
    pd.concat([pio, roic_5y, fcfg_5y, fev_5y, maint_fev_5y, growth_capx_years_5y,
               salesg_5y, roic_ratio, accruals], axis=1)
    .join(fic_by_gv).join(gsector_by_gv)
    .dropna(subset=["piotroski", "roic_5y_avg", "fcf_growth_5y", "fcf_ev_5y_avg"])
    .reset_index().rename(columns={"index": "gvkey"})
    .merge(latest_mcap[["gvkey","market_cap_usd","market_cap_local","price_asof","ttm_dps","adv_usd"]], on="gvkey", how="left")
)

# PE ratio & dividend yield
_latest = panel5.sort_values("fyear").groupby("gvkey").tail(1).set_index("gvkey")
_val = _latest[["ni"]].rename(columns={"ni": "ni_latest"}).reset_index()
metrics = metrics.merge(_val, on="gvkey", how="left")
_ni   = pd.to_numeric(metrics["ni_latest"], errors="coerce")
_mc_m = pd.to_numeric(metrics["market_cap_local"], errors="coerce") / 1e6
_px   = pd.to_numeric(metrics["price_asof"], errors="coerce")
_dps  = pd.to_numeric(metrics["ttm_dps"], errors="coerce")
_pe_ok  = ((_ni > 0) & (_mc_m > 0)).fillna(False)
_div_ok = ((_px > 0) & _dps.notna()).fillna(False)
metrics["pe_ratio"]  = np.where(_pe_ok, _mc_m / _ni.replace(0, np.nan), np.nan)
metrics["div_yield"] = np.where(_div_ok, _dps.clip(lower=0) / _px, np.nan)
metrics["earnings_yield"] = 1.0 / metrics["pe_ratio"].replace([np.inf, -np.inf], np.nan)

# ── Advisory flags ────────────────────────────────────────────────────
metrics["flag_sales_decline"] = (metrics["sales_growth_5y"] < SALES_CAGR_MIN).fillna(True).astype(bool)
metrics["flag_roic_decay"]    = (metrics["roic_ratio_latest_5y"] < ROIC_RATIO_MIN).fillna(True).astype(bool)
metrics["flag_accruals"]      = (metrics["accruals_latest"] > ACCRUALS_MAX).fillna(True).astype(bool)
metrics["flag_cyclical"]      = metrics["gsector"].astype(str).isin(CYCLICAL_GSECTORS)
metrics["flag_nonrecurring"]  = ((metrics["fcf_growth_5y"] > FCF_CAGR_MAX_NORMAL) |
                                 (metrics["roic_ratio_latest_5y"] > ROIC_SURGE_MULT)).fillna(False).astype(bool)
metrics["trap_flags"] = metrics[["flag_sales_decline","flag_roic_decay","flag_accruals"]].sum(axis=1).astype(int)

print(f"Metrics computed for {len(metrics):,} firms")

In [ ]:
# ── 10-year persistence metrics ────────────────────────────────────────
panel_sorted = panel.sort_values(["gvkey", "fyear"])
panel10_all  = panel_sorted.groupby("gvkey").tail(10).copy()
n_fyears     = panel10_all.groupby("gvkey")["fyear"].nunique()
firms_10y    = n_fyears[n_fyears >= 10].index
p10          = panel10_all[panel10_all["gvkey"].isin(firms_10y)].copy()
print(f"Firms with 10+ FYs: {len(firms_10y):,}")

fcf_pos_years_10y = p10.groupby("gvkey")["fcf"].apply(
    lambda s: int((pd.to_numeric(s, errors="coerce") > 0).sum())).rename("fcf_pos_years_10y")

def _cv(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) < 5: return np.nan
    mu = s.mean()
    if abs(mu) < 1e-6: return np.nan
    return s.std() / abs(mu)
fcf_cv_10y = p10.groupby("gvkey")["fcf"].apply(_cv).rename("fcf_cv_10y")

roic_min_10y = p10.groupby("gvkey")["roic"].apply(
    lambda s: pd.to_numeric(s, errors="coerce").min()).rename("roic_min_10y")

# Interest coverage (5y min)
p5from10 = p10.sort_values("fyear").groupby("gvkey").tail(5).copy()
p5from10["_ic"] = pd.to_numeric(p5from10["ebit"], errors="coerce") /                   pd.to_numeric(p5from10["xint"], errors="coerce").where(
                      pd.to_numeric(p5from10["xint"], errors="coerce") > 0)
interest_coverage_5y_min = p5from10.groupby("gvkey")["_ic"].min().rename("interest_coverage_5y_min")
no_int = p5from10.groupby("gvkey")["xint"].apply(lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) <= 0).all())
interest_coverage_5y_min.loc[no_int[no_int].index] = 1e6

# Altman Z'' (EM variant)
latest10 = panel10_all.sort_values("fyear").groupby("gvkey").tail(1).set_index("gvkey")
_wc  = pd.to_numeric(latest10["act"], errors="coerce").fillna(0) - pd.to_numeric(latest10["lct"], errors="coerce").fillna(0)
_at  = pd.to_numeric(latest10["at"], errors="coerce")
_re  = pd.to_numeric(latest10["re"], errors="coerce")
_eb  = pd.to_numeric(latest10["ebit"], errors="coerce")
_ceq = pd.to_numeric(latest10["ceq"], errors="coerce")
_lt  = pd.to_numeric(latest10["lt"], errors="coerce").where(pd.to_numeric(latest10["lt"], errors="coerce") > 0)
altman_z = (6.56*_wc/_at + 3.26*_re/_at + 6.72*_eb/_at + 1.05*_ceq/_lt).rename("altman_z")

# Capital discipline
def _shares_chg(g):
    g = g.sort_values("fyear")
    if len(g) < 6: return np.nan
    c0 = pd.to_numeric(g["csho"].iloc[-6], errors="coerce")
    c1 = pd.to_numeric(g["csho"].iloc[-1], errors="coerce")
    if pd.isna(c0) or pd.isna(c1) or c0 <= 0: return np.nan
    return c1/c0 - 1
shares_chg_5y_pct = p10.groupby("gvkey").apply(_shares_chg).rename("shares_chg_5y_pct")

def _fcf_ni(g):
    g = g.sort_values("fyear").tail(5)
    ni  = pd.to_numeric(g["ni"], errors="coerce")
    fcf = pd.to_numeric(g["fcf"], errors="coerce")
    ok  = (ni > 0) & fcf.notna()
    if ok.sum() < 3: return np.nan
    return (fcf[ok] / ni[ok]).mean()
fcf_ni_ratio_5y_avg = p10.groupby("gvkey").apply(_fcf_ni).rename("fcf_ni_ratio_5y_avg")

def _gm_stability(g):
    g = g.sort_values("fyear").tail(5)
    sale = pd.to_numeric(g["sale"], errors="coerce")
    cogs = pd.to_numeric(g["cogs"], errors="coerce")
    gm = ((sale - cogs) / sale.where(sale > 0)).dropna()
    if len(gm) < 4: return np.nan
    return float(gm.std())
gm_stdev_5y = p10.groupby("gvkey").apply(_gm_stability).rename("gm_stdev_5y")

# Net debt/EBITDA (latest)
_nd  = pd.to_numeric(latest10["net_debt"], errors="coerce")
_ebd = pd.to_numeric(latest10["ebitda"], errors="coerce").where(pd.to_numeric(latest10["ebitda"], errors="coerce") > 0)
net_debt_ebitda = (_nd / _ebd).where(_nd > 0, 0.0).rename("net_debt_ebitda")

# Dividend streak
_sd = sec[["gvkey","datadate","divd"]].copy()
_sd["divd"] = pd.to_numeric(_sd["divd"], errors="coerce").fillna(0)
_sd = _sd[_sd["divd"] > 0]
_sd["year"] = _sd["datadate"].dt.year
ann_dps = _sd.groupby(["gvkey","year"])["divd"].sum().reset_index()
def _streak(g):
    g = g.sort_values("year")
    yrs, vals = g["year"].tolist(), g["divd"].tolist()
    if not yrs: return 0
    streak = 1
    for i in range(len(yrs)-1, 0, -1):
        if yrs[i]-yrs[i-1] == 1 and vals[i] >= vals[i-1]: streak += 1
        else: break
    return streak
div_consecutive_yrs = ann_dps.groupby("gvkey").apply(_streak).rename("div_consecutive_yrs")

# FCF margin (5y avg)
def _fcf_margin(g):
    g = g.sort_values("fyear").tail(5)
    sale = pd.to_numeric(g["sale"], errors="coerce")
    fcf  = pd.to_numeric(g["fcf"], errors="coerce")
    ratio = (fcf / sale.where(sale > 0)).dropna()
    if len(ratio) < 3: return np.nan
    return float(ratio.mean())
fcf_margin_5y_avg = p10.groupby("gvkey").apply(_fcf_margin).rename("fcf_margin_5y_avg")

# ── Merge all persistence metrics ─────────────────────────────────────
_persist = pd.concat([fcf_pos_years_10y, fcf_cv_10y, roic_min_10y, interest_coverage_5y_min,
                      altman_z, shares_chg_5y_pct, fcf_ni_ratio_5y_avg, net_debt_ebitda,
                      div_consecutive_yrs, gm_stdev_5y, fcf_margin_5y_avg], axis=1)
_persist.index.name = "gvkey"
_persist = _persist.reset_index()
_persist["div_consecutive_yrs"] = _persist["div_consecutive_yrs"].fillna(0).astype(int)
metrics = metrics.merge(_persist, on="gvkey", how="left")
print(f"Persistence metrics merged. 10y-history firms in metrics: {metrics['fcf_pos_years_10y'].notna().sum():,}")

## 6 - Hard Gates & Composite Scoring

**12 binary gates** applied sequentially. A firm must pass ALL to enter the ranked universe.

**Composite** = `quality_block^0.6 x persist_block^0.4` where:
- `quality_block` = avg pct-rank of (ROIC 5y, Piotroski)
- `persist_block` = avg of (durability, capital discipline, franchise stability) sub-blocks

In [ ]:
def pct_rank(s, higher_better=True):
    s = pd.to_numeric(s, errors="coerce")
    if not higher_better: s = -s
    return s.rank(pct=True, ascending=True, na_option="top")

m = metrics.copy()
m["fic"] = m["fic"].map(lambda c: ISO3_TO_ISO2.get(c, c))
m = m[m["fic"].isin(UNIVERSE)].copy()
m = m.merge(company[["gvkey", "company_name", "gsubind"]], on="gvkey", how="left")

# CN SOE exclusion
if CN_WHITELIST_PATTERNS:
    whitelist_re = "|".join(CN_WHITELIST_PATTERNS)
    keep_cn = m["company_name"].fillna("").str.contains(whitelist_re, case=False, regex=True)
else:
    keep_cn = pd.Series(False, index=m.index)
is_cn = m["fic"] == "CN"
m = m[~(is_cn & ~keep_cn)].copy()

# Tiered market cap floor
_dm_mask = m["fic"].isin(DM_EX_US)
_em_mask = m["fic"].isin(EM)
m["mcap_floor_usd"] = np.where(_dm_mask, MCAP_MIN_USD_DM, np.where(_em_mask, MCAP_MIN_USD_EM, MCAP_MIN_USD_DM))

# ── 12 HARD GATES ─────────────────────────────────────────────────────
n_pre = len(m)
gates = {
    "10y history":           m["fcf_pos_years_10y"].notna(),
    "FCF pos >= 6/10":       m["fcf_pos_years_10y"].fillna(0) >= 6,
    "netDebt/EBITDA <= 3x":  m["net_debt_ebitda"].fillna(99).le(3.0),
    "IC(5y,min) >= 3x":     m["interest_coverage_5y_min"].fillna(-1).ge(3.0),
    "Altman Z'' >= 1.1":    m["altman_z"].fillna(-999).ge(1.1),
    "Market cap floor":     m["market_cap_usd"].fillna(0) >= m["mcap_floor_usd"],
    "ADV >= $100k":         m["adv_usd"].fillna(0) >= ADV_MIN_USD,
    "ROIC 5y avg >= 10%":   m["roic_5y_avg"].fillna(-1).ge(ROIC_5Y_MIN),
    "ROIC 10y min >= 5%":   m["roic_min_10y"].fillna(-1).ge(ROIC_10Y_MIN),
    "FCF margin 5y >= 5%":  m["fcf_margin_5y_avg"].fillna(-1).ge(FCF_MARGIN_5Y_MIN),
    "Piotroski >= 5":       m["piotroski"].fillna(-1).ge(PIOTROSKI_MIN),
    "Valuation (4-way OR)": ((m["earnings_yield"].fillna(-999) > EY_MIN) |
                             (m["fcf_ev_5y_avg"].fillna(-999) > FCF_EV_MIN) |
                             (m["maint_fcf_ev_5y_avg"].fillna(-999) > FCF_EV_MIN) |
                             ((m["roic_5y_avg"].fillna(-1) >= ROIC_ESCAPE_MIN) &
                              (m["earnings_yield"].fillna(-999) > EY_ESCAPE_MIN))),
}

print(f"Starting universe: {n_pre:,}")
cum = pd.Series(True, index=m.index)
for label, g in gates.items():
    cum = cum & g
    print(f"  {label:35s} -> {int(cum.sum()):,} remaining")

m = m[cum].copy()
print(f"\nPassing ALL gates: {len(m):,}")

# ── COMPOSITE SCORING ──────────────────────────────────────────────────
m["quality_block"] = ((pct_rank(m["roic_5y_avg"], True) + pct_rank(m["piotroski"], True)) / 2).clip(lower=0.1)

m["durability_block"] = ((pct_rank(m["fcf_pos_years_10y"], True) + pct_rank(m["fcf_cv_10y"], False) +
                          pct_rank(m["roic_min_10y"], True) + pct_rank(m["interest_coverage_5y_min"], True)) / 4).clip(lower=0.1)

m["capital_block"] = ((pct_rank(m["shares_chg_5y_pct"], False) + pct_rank(m["fcf_ni_ratio_5y_avg"], True) +
                       pct_rank(m["div_consecutive_yrs"], True)) / 3).clip(lower=0.1)

m["franchise_block"] = ((pct_rank(m["gm_stdev_5y"], False) + pct_rank(m["sales_growth_5y"], True)) / 2).clip(lower=0.1)

m["persist_block"] = ((m["durability_block"] + m["capital_block"] + m["franchise_block"]) / 3).clip(lower=0.1)
m["composite"] = m["quality_block"] ** 0.6 * m["persist_block"] ** 0.4

_blocks4 = m[["quality_block","durability_block","capital_block","franchise_block"]].rename(
    columns={"quality_block":"quality","durability_block":"durability","capital_block":"capital","franchise_block":"franchise"})
m["driver"] = _blocks4.idxmax(axis=1)
m["rank_composite"] = m["composite"].rank(ascending=False, method="min").astype(int)
m = m.sort_values("rank_composite").reset_index(drop=True)
print(f"Composite ranked: {len(m):,} names")

## 7 - Structural Under-Pricing Tags

Seven binary tags identify names that may be structurally under-priced due to non-fundamental reasons:

| Tag | Rationale |
|-----|-----------|
| ESG excluded | Sector exclusion by ESG mandates (tobacco, gambling, weapons) |
| Benchmark orphan | Too small for major indices, too large for micro-cap |
| Low coverage | Below analyst coverage threshold |
| Complexity | Holding structures, dual-class shares |
| Style orphan | Neither pure value nor pure growth |
| Orphan country | Under-allocated EM/frontier domicile |
| Retail dominated | Markets with high retail ownership distortion |

Scoring: `structural_multiplier = 1 + 0.20 x pure_tags + 0.10 x mixed_tags`

In [ ]:
m["_gsubind_int"] = pd.to_numeric(m["gsubind"], errors="coerce").astype("Int64")
m["_gsector_int"] = pd.to_numeric(m["gsector"], errors="coerce").astype("Int64")

# (1) ESG excluded
_esg_all = STRUCT_ESG_DEEP | STRUCT_ESG_PARTIAL
m["tag_esg_excluded"] = m["_gsubind_int"].isin(_esg_all).astype(int)

# (2) Benchmark orphan
def _bmk_orphan(row):
    cap = row["market_cap_usd"]
    if pd.isna(cap): return 0
    if row["fic"] in STRUCT_DM_COUNTRIES:
        return int(STRUCT_BMK_DM_LO < cap < STRUCT_BMK_DM_HI)
    else:
        return int(STRUCT_BMK_EM_LO < cap < STRUCT_BMK_EM_HI)
m["tag_benchmark_orphan"] = m.apply(_bmk_orphan, axis=1)

# (3) Low coverage (mcap proxy — no IBES in pipeline)
m["tag_low_coverage"] = (m["market_cap_usd"].fillna(0) < STRUCT_COVERAGE_MCAP_FLOOR).astype(int)

# (4) Complexity
name_has_holding = m["company_name"].fillna("").str.upper().str.contains(r"\bHOLDING", regex=True, na=False)
not_fin_re = ~m["_gsector_int"].isin([40, 60])
try:
    _gvs = ",".join(f"'{g}'" for g in m["gvkey"].unique())
    q_iid = f"""SELECT gvkey, COUNT(DISTINCT iid) AS n_issues FROM comp.g_secd
               WHERE gvkey IN ({_gvs}) AND datadate BETWEEN '2023-01-01' AND '{SCREEN_ASOF}'
               GROUP BY gvkey"""
    dual_df = sql_cached(q_iid, "g_secd_iid_count_v1")
    _iid_map = dual_df.set_index("gvkey")["n_issues"]
    cond_c = m["gvkey"].map(_iid_map).fillna(1).astype(int) >= 2
except Exception:
    cond_c = pd.Series(False, index=m.index)
m["tag_complexity"] = ((m["_gsubind_int"] == 40201040) | (name_has_holding & not_fin_re) | cond_c).astype(int)

# (5) Style orphan (neither value nor growth)
def _z(s):
    mu, sd = s.mean(skipna=True), s.std(skipna=True)
    if sd == 0 or pd.isna(sd): return pd.Series(0.0, index=s.index)
    return (s - mu) / sd
_ey   = pd.to_numeric(m["earnings_yield"], errors="coerce")
_pe   = pd.to_numeric(m["pe_ratio"], errors="coerce")
_salg = pd.to_numeric(m["sales_growth_5y"], errors="coerce")
_roic = pd.to_numeric(m["roic_5y_avg"], errors="coerce")
style_pct = (_z(_ey) + _z(-_pe) - _z(_salg) - _z(_roic)).rank(pct=True, na_option="top")
m["tag_style_orphan"] = ((style_pct >= 0.40) & (style_pct <= 0.60)).astype(int)

# (6) Orphan country
m["tag_orphan_country"] = m["fic"].isin(STRUCT_ORPHAN_COUNTRY).astype(int)

# (7) Retail dominated
m["tag_retail_dominated"] = m["fic"].isin(STRUCT_RETAIL_DOMINATED).astype(int)

m = m.drop(columns=["_gsubind_int", "_gsector_int"], errors="ignore")

# ── Structural scoring ────────────────────────────────────────────────
m["structural_score_pure"] = (m["tag_esg_excluded"] + m["tag_benchmark_orphan"] +
                              m["tag_low_coverage"] + m["tag_complexity"] + m["tag_style_orphan"])
m["structural_score_mixed"] = m["tag_orphan_country"] + m["tag_retail_dominated"]
m["structural_score"] = m["structural_score_pure"] + 0.5 * m["structural_score_mixed"]

_TAG_NAMES = [("tag_esg_excluded","ESG"), ("tag_benchmark_orphan","BMK"), ("tag_low_coverage","COV"),
              ("tag_complexity","CPX"), ("tag_style_orphan","STY"), ("tag_orphan_country","CTY"),
              ("tag_retail_dominated","RTL")]
def _tag_string(row):
    bits = [code for col, code in _TAG_NAMES if row[col]]
    return "|".join(bits) if bits else "-"
m["structural_tags"] = m.apply(_tag_string, axis=1)

print(f"Structural tags computed. Score >= 2: {int((m['structural_score'] >= 2).sum())} names")
_tag_cols = [c for c, _ in _TAG_NAMES]
for c in _tag_cols:
    print(f"  {c:28s} {int(m[c].sum()):3d} / {len(m)}")

## 8 - Final Selection (Top 20)

1. Compute `final_score = composite x structural_multiplier`
2. Apply 9 advisory flags (SRACNDLXV)
3. Select top 20 zero-flag names with dual (country + sector) concentration caps

In [ ]:
# ── Final score ────────────────────────────────────────────────────────
m["structural_multiplier"] = (1.0 + STRUCT_PURE_WEIGHT * m["structural_score_pure"]
                                  + STRUCT_MIXED_WEIGHT * m["structural_score_mixed"])
m["final_score"] = m["composite"] * m["structural_multiplier"]
m = m.sort_values("final_score", ascending=False).reset_index(drop=True)
m["rank_final"] = range(1, len(m)+1)

# ── Advisory flags on gate-passers ─────────────────────────────────────
_stale_info = latest_mcap[["gvkey", "asof_price_date"]].copy()
_stale_info["_flag_stale"] = (pd.to_datetime(SCREEN_ASOF) - pd.to_datetime(_stale_info["asof_price_date"])).dt.days > 60
m = m.merge(_stale_info[["gvkey", "_flag_stale"]], on="gvkey", how="left")
m["flag_stale"]          = m["_flag_stale"].fillna(False).astype(bool)
m.drop(columns=["_flag_stale"], inplace=True)
m["flag_illiquid"]       = (m["adv_usd"].fillna(0) < ADV_FLAG_THRESHOLD).astype(bool)
m["flag_dilution"]       = (m["shares_chg_5y_pct"].fillna(0) > SHARES_DILUTION_FLAG).astype(bool)
m["flag_fcf_ev_extreme"] = (m["fcf_ev_5y_avg"].fillna(0) > FCF_EV_FLAG_CEIL).astype(bool)
m["flag_growth_capex"]   = (m["growth_capx_years_5y"].fillna(0) >= 3).astype(bool)

_WARNING_FLAGS = ["flag_sales_decline","flag_roic_decay","flag_accruals",
                  "flag_cyclical","flag_nonrecurring","flag_stale",
                  "flag_illiquid","flag_dilution","flag_fcf_ev_extreme"]
m["n_flags"] = m[_WARNING_FLAGS].sum(axis=1).astype(int)

print(f"Gate-passers: {len(m)}  |  Zero-flag (clean): {int((m['n_flags']==0).sum())}  |  Flagged: {int((m['n_flags']>0).sum())}")

# ── Selection loop (zero-flag only, dual caps) ─────────────────────────
m_pool = m[m["n_flags"] == 0].copy()
country_max = max(int(N_FINAL * COUNTRY_MAX_PCT), 1)
sector_max  = max(int(N_FINAL * SECTOR_MAX_PCT), 1)
country_cnt, sector_cnt = {}, {}
m["pick_status"] = "flagged"
m.loc[m["n_flags"] == 0, "pick_status"] = "unreached"
picked_idx = []

for idx, row in m_pool.iterrows():
    c, s = row["fic"], str(row.get("gsector", ""))
    if country_cnt.get(c, 0) >= country_max:
        m.at[idx, "pick_status"] = "skip_country"; continue
    if sector_cnt.get(s, 0) >= sector_max:
        m.at[idx, "pick_status"] = "skip_sector"; continue
    m.at[idx, "pick_status"] = "picked"
    picked_idx.append(idx)
    country_cnt[c] = country_cnt.get(c, 0) + 1
    sector_cnt[s]  = sector_cnt.get(s, 0) + 1
    if len(picked_idx) >= N_FINAL: break

selected = m.loc[picked_idx].reset_index(drop=True)
print(f"\nSelected: {len(selected)}/{N_FINAL}  |  Country cap: {country_max}  |  Sector cap: {sector_max}")
if len(selected) < N_FINAL:
    print(f"WARNING: only filled {len(selected)}/{N_FINAL} slots")

## 9 - Selected Portfolio & Full Ranking

In [ ]:
# ── Flag string (SRACNDLXV) ────────────────────────────────────────────
_FLAG_MAP = [
    ("flag_sales_decline","S"), ("flag_roic_decay","R"), ("flag_accruals","A"),
    ("flag_cyclical","C"), ("flag_nonrecurring","N"), ("flag_stale","D"),
    ("flag_illiquid","L"), ("flag_dilution","X"), ("flag_fcf_ev_extreme","V"),
]
def _flag_str(row):
    return "".join(code if row.get(col, False) else "." for col, code in _FLAG_MAP)

# TTM ROIC
_ttm_roic = (panel.sort_values("datadate").groupby("gvkey").tail(1)
             .set_index("gvkey")[["roic"]].rename(columns={"roic": "roic_ttm"}))

# ── Selected portfolio table ──────────────────────────────────────────
show = selected.copy()
show = show.merge(_ttm_roic, left_index=True, right_index=True, how="left")
show["flags_str"] = show.apply(_flag_str, axis=1)
disp = pd.DataFrame({
    "#":        range(1, len(show)+1),
    "Country":  show["fic"].values,
    "Company":  show["company_name"].str.slice(0, 30).values,
    "Sector":   show["gsector"].astype(str).values,
    "Mcap$m":   (show["market_cap_usd"]/1e6).round(0).astype(int).values,
    "Score":    show["final_score"].round(3).values,
    "Comp":     show["composite"].round(3).values,
    "Qual":     show["quality_block"].round(2).values,
    "Persist":  show["persist_block"].round(2).values,
    "Driver":   show["driver"].values,
    "ROIC_ttm": (show["roic_ttm"]*100).round(1).values,
    "ROIC_5y":  (show["roic_5y_avg"]*100).round(1).values,
    "FCFm%":    (show["fcf_margin_5y_avg"]*100).round(1).values,
    "Pio":      show["piotroski"].astype(int).values,
    "EY%":      (show["earnings_yield"]*100).round(2).values,
    "Flags":    show["flags_str"].values,
    "Struct":   show["structural_tags"].values,
})
print(f"=== SELECTED PORTFOLIO ({len(disp)} names) ===\n")
print(tabulate(disp, headers="keys", tablefmt="simple", showindex=False))

# ── Full gate-passer ranking ──────────────────────────────────────────
fv = m.copy()
fv = fv.merge(_ttm_roic, left_index=True, right_index=True, how="left")
fv["flags_str"] = fv.apply(_flag_str, axis=1)
full_disp = pd.DataFrame({
    "Rk":      fv["rank_final"].values,
    "Status":  fv["pick_status"].values,
    "Ctry":    fv["fic"].values,
    "Company": fv["company_name"].str.slice(0, 28).values,
    "Sec":     fv["gsector"].astype(str).values,
    "Score":   fv["final_score"].round(3).values,
    "Comp":    fv["composite"].round(3).values,
    "ROIC5y":  (fv["roic_5y_avg"]*100).round(1).values,
    "Pio":     fv["piotroski"].astype(int).values,
    "EY%":     (fv["earnings_yield"]*100).round(2).values,
    "Flags":   fv["flags_str"].values,
    "nF":      fv["n_flags"].values,
})
print(f"\n=== ALL GATE-PASSERS ({len(full_disp)} names) ===")
print(f"Status: picked | skip_country | skip_sector | unreached | flagged\n")
print(tabulate(full_disp, headers="keys", tablefmt="simple", showindex=False))

## 10 - Two-Year Survivability Stress Test

Three scenarios applied to latest annual financials:

| Scenario | Revenue | Margin | Rate bump | CapEx | Dividend |
|----------|---------|--------|-----------|-------|----------|
| Base     | x1.00   | x1.00  | +0bp      | x1.00 | held     |
| Stress   | x0.80   | x0.70  | +200bp    | x1.00 | held     |
| Severe   | x0.65   | x0.50  | +400bp    | x0.70 | cut 50%  |

Per-scenario grading: **Fail** (IC<1.5x, ND/EB>5x, FCF<0+runway<1y, Z<1.1) | **Marginal** | **Pass**

Overall: pass/pass = *highly_resilient*, pass/marginal = *stress_resilient*, etc.

In [ ]:
# ── Build stress inputs ────────────────────────────────────────────────
_pan = panel.sort_values(["gvkey", "datadate"]).copy()
_cols = ["sale","ebitda","ebit","xint","capx","fcf","dvt","tax_rate","che","dlc","dltt","at","ceq","lt"]
_ttm_stress = (_pan.groupby("gvkey").tail(1).set_index("gvkey")[_cols]
               .rename(columns={"sale":"rev", "che":"cash"}))
_ttm_stress["total_debt"] = _ttm_stress["dlc"].fillna(0) + _ttm_stress["dltt"].fillna(0)

def _hist(g):
    g = g.sort_values("datadate").tail(10)
    last5 = g.tail(5)
    margins = last5["ebitda"] / last5["sale"].replace(0, np.nan)
    return pd.Series({"ebitda_margin_5y_avg": margins.mean()})

_hst = _pan.groupby("gvkey").apply(_hist)
stress_inputs = _ttm_stress.join(_hst).reset_index()
_ctx = m[["gvkey","company_name","fic","gsector","market_cap_usd","altman_z"]]
stress_inputs = stress_inputs.merge(_ctx, on="gvkey", how="inner")
stress_inputs["in_portfolio"] = stress_inputs["gvkey"].isin(selected["gvkey"])

# ── Scenarios ──────────────────────────────────────────────────────────
SCENARIOS = {
    "base":   dict(rev=1.00, margin=1.00, dr=0.0, capex=1.00, div_cut=0.00),
    "stress": dict(rev=0.80, margin=0.70, dr=0.02, capex=1.00, div_cut=0.00),
    "severe": dict(rev=0.65, margin=0.50, dr=0.04, capex=0.70, div_cut=0.50),
}

def _safe_div_arr(num, den):
    num, den = np.asarray(num, dtype=float), np.asarray(den, dtype=float)
    return num / np.where(den == 0, np.nan, den)

def _run_scenario(df, s):
    def _f(c): return pd.to_numeric(df[c], errors="coerce").astype(float)
    rev, ebitda, ebit = _f("rev"), _f("ebitda"), _f("ebit")
    xint, capx, dvt = _f("xint"), _f("capx"), _f("dvt")
    tax_rate, cash, total_debt = _f("tax_rate"), _f("cash"), _f("total_debt")
    at_, margin_hist, z_base = _f("at"), _f("ebitda_margin_5y_avg"), _f("altman_z")

    rev_s = rev * s["rev"]
    cur_margin = _safe_div_arr(ebitda, rev)
    m_base = margin_hist.where(margin_hist.notna() & (margin_hist > 0), cur_margin)
    ebitda_s = rev_s * m_base * s["margin"]
    da = (ebitda - ebit).fillna(0)
    ebit_s = ebitda_s - da
    kd = pd.Series(_safe_div_arr(xint, total_debt), index=df.index).fillna(0).clip(0, 0.25)
    int_s = total_debt * (kd + s["dr"])
    pretax = ebit_s - int_s
    tax_s = np.where(pretax > 0, pretax * tax_rate.fillna(0.25), 0.0)
    capex_s = capx.fillna(0) * s["capex"]
    div_s = dvt.fillna(0) * (1 - s["div_cut"])
    fcf_s = ebitda_s - tax_s - int_s - capex_s - 0.05*(rev_s - rev)

    ic_s = np.where(int_s > 0, _safe_div_arr(ebit_s, int_s), np.inf)
    nd_eb_s = np.where(ebitda_s > 0, _safe_div_arr(total_debt - cash, ebitda_s), np.inf)
    div_cov_s = np.where(div_s > 0, _safe_div_arr(fcf_s, div_s), np.inf)
    draw = np.where(fcf_s >= div_s, 0.0, div_s - fcf_s)
    runway_s = np.where(draw > 0, _safe_div_arr(cash, draw), np.inf)
    z_s = z_base + 6.72 * _safe_div_arr(ebit_s - ebit, at_)

    return pd.DataFrame({"ic":ic_s, "nd_eb":nd_eb_s, "fcf":np.asarray(fcf_s,dtype=float),
                         "div_cov":div_cov_s, "runway":runway_s, "z":np.asarray(z_s,dtype=float)}, index=df.index)

out = stress_inputs[["gvkey","company_name","fic","gsector","market_cap_usd","in_portfolio"]].copy()
for nm, s in SCENARIOS.items():
    res = _run_scenario(stress_inputs, s)
    for c in res.columns:
        out[f"{nm}_{c}"] = res[c].values

# ── Rating logic ───────────────────────────────────────────────────────
def _rate_row(ic, nde, fcf, div_cov, runway, z):
    ic = np.inf if pd.isna(ic) else ic
    nde = -np.inf if pd.isna(nde) else nde
    div_cov = np.inf if pd.isna(div_cov) else div_cov
    runway = np.inf if pd.isna(runway) else runway
    z = 999 if pd.isna(z) else z
    fcf = 0 if pd.isna(fcf) else fcf
    if ic < 1.5 or nde > 5 or (fcf < 0 and runway < 1) or z < 1.1: return "fail"
    if ic < 3 or nde > 3 or div_cov < 1 or runway < 3 or fcf <= 0: return "marginal"
    return "pass"

for sc in ("base","stress","severe"):
    out[f"rate_{sc}"] = out.apply(lambda r: _rate_row(
        r[f"{sc}_ic"], r[f"{sc}_nd_eb"], r[f"{sc}_fcf"],
        r[f"{sc}_div_cov"], r[f"{sc}_runway"], r[f"{sc}_z"]), axis=1)

def _overall(r):
    s, v = r["rate_stress"], r["rate_severe"]
    if s == "fail": return "fail"
    if s == "marginal": return "marginal"
    if v == "pass": return "highly_resilient"
    if v == "marginal": return "stress_resilient"
    return "severe_vulnerable"
out["overall"] = out.apply(_overall, axis=1)
out["_ov_rk"] = out["overall"].map({"highly_resilient":1,"stress_resilient":2,"marginal":3,"severe_vulnerable":4,"fail":5})
stress_results = out

_order = ["highly_resilient","stress_resilient","marginal","severe_vulnerable","fail"]
print("\nStress bucket distribution (all gate-passers):")
print(stress_results["overall"].value_counts().reindex(_order).fillna(0).astype(int).to_string())
print("\nPortfolio stress buckets:")
print(stress_results[stress_results["in_portfolio"]]["overall"].value_counts().reindex(_order).fillna(0).astype(int).to_string())

In [ ]:
# ── Per-name stress breakdown (portfolio) ──────────────────────────────
port = stress_results[stress_results["in_portfolio"]].copy()
port = port.sort_values(["_ov_rk", "market_cap_usd"], ascending=[True, False])

def _fmt(x, d=1):
    if pd.isna(x) or not np.isfinite(x): return "inf"
    return f"{x:.{d}f}"

tbl = pd.DataFrame({
    "Country":  port["fic"].values,
    "Company":  port["company_name"].str.slice(0, 26).values,
    "Stress":   port["rate_stress"].values,
    "Severe":   port["rate_severe"].values,
    "Overall":  port["overall"].values,
    "IC_s":     port["stress_ic"].apply(_fmt).values,
    "ND/EB_s":  port["stress_nd_eb"].apply(_fmt).values,
    "DivCov_s": port["stress_div_cov"].apply(_fmt).values,
    "Z_s":      port["stress_z"].apply(_fmt).values,
})
print(f"\n=== STRESS TEST — Portfolio ({len(port)} names) ===\n")
print(tabulate(tbl, headers="keys", tablefmt="simple", showindex=False))

# Deployment rules
_DEPLOY = {"highly_resilient":"Core; add on drawdown", "stress_resilient":"Hold; no drawdown adds",
           "severe_vulnerable":"Size cap; review", "marginal":"Review; no adds", "fail":"Exclude"}
flagged_stress = port[port["overall"].isin(["marginal","severe_vulnerable","fail"])]
if len(flagged_stress):
    print(f"\nREVIEW: {len(flagged_stress)} name(s) below top-two stress buckets")

## 11 - Combined Ranking: Composite x Stress Resilience

In [ ]:
# ── Cross-tab & best-of-both shortlist ─────────────────────────────────
_mcols = [c for c in ["gvkey","rank_final","final_score","composite","structural_tags",
          "pick_status","quality_block","persist_block","driver","roic_5y_avg",
          "fcf_margin_5y_avg","fcf_ev_5y_avg","earnings_yield","altman_z","net_debt_ebitda",
          "company_name","fic","gsector"] if c in m.columns]
_scols = [c for c in ["gvkey","overall","_ov_rk","rate_stress","rate_severe","in_portfolio"] if c in stress_results.columns]
combo = m[_mcols].merge(stress_results[_scols], on="gvkey", how="left")
combo["rank_decile"] = pd.qcut(combo["rank_final"], 10, labels=range(1,11), duplicates="drop").astype(int)

_buckets = ["highly_resilient","stress_resilient","marginal","severe_vulnerable","fail"]
ct = combo.groupby(["overall","rank_decile"]).size().unstack(fill_value=0).reindex(_buckets, fill_value=0)
print("=== CROSS-TAB: stress bucket x composite-rank decile ===")
print("(decile 1 = top composite; bucket: best -> worst)\n")
print(ct.to_string())

# Best-of-both: top 50 by composite AND stress-safe
TOP_N_RANK = 50
SAFE_BUCKETS = {"highly_resilient","stress_resilient"}
short = combo[(combo["rank_final"] <= TOP_N_RANK) & (combo["overall"].isin(SAFE_BUCKETS))].copy()
short = short.sort_values(["_ov_rk","rank_final"])

print(f"\n=== BEST-OF-BOTH — top {TOP_N_RANK} composite + stress-safe ({len(short)} names) ===\n")
short_disp = pd.DataFrame({
    "Rk":     short["rank_final"].values,
    "Ctry":   short["fic"].values,
    "Company": short["company_name"].str.slice(0,28).values,
    "Score":  short["final_score"].round(3).values,
    "Stress": short["overall"].values,
    "ROIC5y": (short["roic_5y_avg"]*100).round(1).values,
    "FCFm%":  (short["fcf_margin_5y_avg"]*100).round(1).values,
    "EY%":    (short["earnings_yield"]*100).round(2).values,
})
print(tabulate(short_disp, headers="keys", tablefmt="simple", showindex=False))

## 12 - Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0, 0].hist(metrics["fcf_ev_5y_avg"].clip(-0.2, 0.5), bins=40, color="#365f91")
axes[0, 0].set_title("5y-avg FCF/EV — Tier-1 Universe")
axes[0, 0].set_xlabel("FCF / EV")

axes[0, 1].hist(metrics["piotroski"], bins=range(0, 11), color="#93c47d", align="left")
axes[0, 1].set_title("Piotroski F-score Distribution")
axes[0, 1].set_xlabel("F-score (0-9)")

cc = selected["fic"].value_counts()
axes[1, 0].bar(cc.index, cc.values, color="#cc4125")
axes[1, 0].set_title("Final Portfolio — Country Breakdown")
axes[1, 0].tick_params(axis="x", rotation=45)

sc = selected["gsector"].astype(str).value_counts()
axes[1, 1].bar(sc.index, sc.values, color="#674ea7")
axes[1, 1].set_title("Final Portfolio — GICS Sector")

plt.tight_layout()
plt.savefig(OUT / "screen_diagnostics.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── Top-10 bar charts ──────────────────────────────────────────────────
top10 = selected.head(10).copy()
top10["short_name"] = top10["company_name"].str[:18]

metric_cols = {"piotroski":"Piotroski F", "roic_5y_avg":"5y Avg ROIC",
               "sales_growth_5y":"5y Sales CAGR", "fcf_ev_5y_avg":"5y Avg FCF/EV", "composite":"Composite"}
palette = ["#365f91","#93c47d","#cc4125","#674ea7","#e69138"]

fig, axes = plt.subplots(1, len(metric_cols), figsize=(18, 6.5), sharey=False)
fig.suptitle("Top 10 — Key Screen Metrics", fontsize=14, fontweight="bold", y=1.02)
for i, (col, label) in enumerate(metric_cols.items()):
    ax = axes[i]
    vals = top10[col].astype(float)
    if col in ("roic_5y_avg","sales_growth_5y","fcf_ev_5y_avg"):
        vals = vals * 100; label += " (%)"
    ax.barh(top10["short_name"], vals, color=palette[i], edgecolor="white")
    ax.set_title(label, fontsize=10)
    ax.invert_yaxis()
    ax.tick_params(axis="y", labelsize=8)
    if i > 0: ax.set_yticklabels([])
plt.tight_layout()
plt.savefig(OUT / "top10_metrics.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── Radar chart ────────────────────────────────────────────────────────
from math import pi as PI

radar_cols = ["piotroski","roic_5y_avg","sales_growth_5y","fcf_ev_5y_avg","composite"]
radar_labels = ["Piotroski","ROIC 5y","Sales Growth 5y","FCF/EV 5y","Composite"]
normed = top10[radar_cols].apply(lambda c: (c - c.min()) / (c.max() - c.min() + 1e-9))
N_rc = len(radar_cols)
angles = [n / float(N_rc) * 2 * PI for n in range(N_rc)] + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(PI / 2); ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(radar_labels, fontsize=9)

cmap = plt.cm.tab10
for idx, (_, row) in enumerate(normed.iterrows()):
    vals = row[radar_cols].tolist() + [row[radar_cols[0]]]
    ax.plot(angles, vals, linewidth=1.8, label=top10["short_name"].iloc[idx], color=cmap(idx%10))
    ax.fill(angles, vals, alpha=0.06, color=cmap(idx%10))

ax.set_title("Top 10 — Normalised Metric Radar", fontsize=13, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.45, 1.1), fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "top10_radar.png", dpi=160, bbox_inches="tight")
plt.show()

## 13 - Persist Outputs

In [ ]:
# Save key outputs
selected.to_csv(OUT / "portfolio_screen_final20.csv", index=False)
m.to_csv(OUT / "gate_passers_full.csv", index=False)
stress_results.to_csv(OUT / "stress_results.csv", index=False)
print(f"Outputs saved to {OUT}/")
print(f"  portfolio_screen_final20.csv  ({len(selected)} rows)")
print(f"  gate_passers_full.csv         ({len(m)} rows)")
print(f"  stress_results.csv            ({len(stress_results)} rows)")